import all libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import TimestampType

Find the external location

In [0]:
%sql
describe external location `control_datalake_processed`

Extract the destination URL

In [0]:
processed_dir = spark.sql("describe external location `control_datalake_processed`").select("url").collect()[0][0]
print(processed_dir)

Load the raw table information

In [0]:
raw_synthetic = spark.read.table("raw_control_datalake.dev.raw_synthetic_data")
raw_data_overview = spark.read.table("raw_control_datalake.dev.raw_data_overview")
raw_data_dictionary = spark.read.table("raw_control_datalake.dev.raw_data_dictionary")
raw_data_anomaly = spark.read.table("raw_control_datalake.dev.raw_data_anomaly")
raw_control_logic = spark.read.table("raw_control_datalake.dev.raw_control_logic")


implement the required transformations for silver

Insert time ingestion

Add the ingestion time

In [0]:
raw_synthetic_df = raw_synthetic.withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_overview_df = raw_data_overview.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_dictionary_df = raw_data_dictionary.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_data_anomaly_df = raw_data_anomaly.drop("_id").withColumn("ingestion_timestamp", F.current_timestamp())\
                    .withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))
raw_control_logic_df = raw_control_logic.withColumn("modified_timestamp", F.lit(None).cast(TimestampType()))

convert the ingestion time to string ("This is required to push the data via Fast API")

In [0]:
raw_synthetic_df = raw_synthetic_df.withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("string"))\
                    .withColumn("modified_timestamp", F.lit(None).cast("string"))
raw_data_overview_df = raw_data_overview_df.withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("string"))\
                    .withColumn("modified_timestamp", F.lit(None).cast("string"))

raw_data_dictionary_df = raw_data_dictionary_df.withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("string"))\
                    .withColumn("modified_timestamp", F.lit(None).cast("string"))

raw_data_anomaly_df = raw_data_anomaly_df.withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("string"))\
                    .withColumn("modified_timestamp", F.lit(None).cast("string"))
raw_control_logic_df = raw_control_logic_df.withColumn("modified_timestamp", F.col("modified_timestamp").cast("string"))\
                        .withColumn("created_timestamp", F.col("created_timestamp").cast("string"))


In [0]:
raw_synthetic_df.printSchema()

In [0]:
catalog = "process_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")
raw_synthetic_df.write.mode("overwrite").saveAsTable("processed_synthetic_data")
raw_data_overview_df.write.mode("overwrite").saveAsTable("processed_data_overview")
raw_data_dictionary_df.write.mode("overwrite").saveAsTable("processed_data_dictionary")
raw_data_anomaly_df.write.mode("overwrite").saveAsTable("processed_data_anomaly")



create the silver control logic table

In [0]:
catalog = "process_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")
raw_control_logic_df.write.mode("overwrite").saveAsTable("processed_control_logic")

In [0]:
catalog = "process_control_datalake"
database = "dev"
spark.sql(f"create database if not exists {catalog}.{database}")
spark.sql(f"use {catalog}.{database}")

In [0]:
logic = spark.sql("select Control_logic from processed_control_logic").collect()[0][0]
print(logic)

### Running the control Logic

In [0]:
%sql 
--DROP TABLE IF EXISTS raw_control_datalake.dev.raw_control_logic

set south african timezone

In [0]:

control_exception = spark.sql(f"""{logic}""")
control_exception = control_exception.withColumn("detection_time", F.current_timestamp())
control_exception = control_exception.withColumn("detection_time", F.col("detection_time").cast("string"))


In [0]:
display(control_exception)

In [0]:
control_exception.printSchema()

### Save the exception as a table

In [0]:
%sql
SHOW SCHEMAS

In [0]:
#spark.sql(f""" drop table if exists processed_control_exception""")

In [0]:
control_exception.write.mode("overwrite").saveAsTable("processed_control_exception")